# ETF Explorer (v1)

Discovery view over Stashaway's full ETF Explorer offering (~98 ETFs).
Computes multi-window metrics (1Y/3Y/5Y) + correlation with your combined book.
Renders to `reports/etf_explorer.html`.

In [1]:
from datetime import date
from pathlib import Path

from hailmary.allocation.book_config import MGMT_FEES_ANNUAL, ROLES
from hailmary.allocation.etf_explorer import build_etf_explorer, render_etf_explorer_report
from hailmary.allocation.portfolios import from_parsed
from hailmary.allocation.statements import parse_statement
from hailmary.allocation.returns import last_business_day_on_or_before
from hailmary.data.providers import YahooFinanceProvider

ETF_XLSX = Path('../../data/stashaway_etf_universe.xlsx')
STATEMENT_PATH = Path('../../data/statements/2026-04 StashAway Monthly Statement.pdf')
REPORT_PATH = Path('../../reports/etf_explorer.html')
START = date(2020, 1, 1)
END = last_business_day_on_or_before(date.today())
TARGET_ANN_RETURN = 0.05
print(f'window: {START}..{END}')

window: 2020-01-01..2026-05-22


## Load user's book (for correlation reference)

In [2]:
parsed = parse_statement(STATEMENT_PATH)
portfolios = [
    from_parsed(
        p,
        roles=ROLES[p.name],
        metadata={'management_fee_annual': MGMT_FEES_ANNUAL.get(p.name, 0.0)},
    )
    for p in parsed if p.name in ROLES
]
provider = YahooFinanceProvider()
fx_bars = provider.get_bars(['USDSGD=X'], START, END)
fx_series_usd_sgd = fx_bars.xs('USDSGD=X', level=0)['close']
print(f'{len(portfolios)} portfolios loaded')

2026-05-24 23:46:48.323 | DEBUG    | hailmary.allocation.statements:get:169 - Statement cache hit for 2026-04 StashAway Monthly Statement.pdf


2026-05-24 23:46:48.325 | INFO     | hailmary.data.providers.yahoo:get_bars:64 - Fetching 1 symbols from Yahoo (2020-01-01 → 2026-05-22); inclusive


2026-05-24 23:46:48.831 | DEBUG    | hailmary.data.cache:set:45 - Cached 1663 rows key=850eb4611058


15 portfolios loaded


## Build the explorer DataFrame

In [3]:
explorer_df = build_etf_explorer(
    ETF_XLSX,
    portfolios=portfolios,
    price_source=provider,
    fx_series_usd_sgd=fx_series_usd_sgd,
    start=START,
    end=END,
)
print(f'{len(explorer_df)} ETFs, {explorer_df["has_data"].sum()} with Yahoo data')
explorer_df.head(20)

2026-05-24 23:46:49.056 | INFO     | hailmary.allocation.etf_explorer:build_etf_explorer:153 - ETF Explorer: fetching 98 symbols from Yahoo…


2026-05-24 23:46:49.056 | INFO     | hailmary.data.providers.yahoo:get_bars:64 - Fetching 98 symbols from Yahoo (2020-01-01 → 2026-05-22); inclusive


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: XAID"}}}


$SDIG: possibly delisted; no timezone found


$XAID: possibly delisted; no timezone found


...

2026-05-24 23:47:03.178 | DEBUG    | hailmary.data.cache:get:38 - Cache hit key=ddbd4353781a


2026-05-24 23:47:03.202 | INFO     | hailmary.data.providers.yahoo:get_bars:64 - Fetching 2 symbols from Yahoo (2020-01-01 → 2026-05-22); inclusive


2026-05-24 23:47:03.267 | DEBUG    | hailmary.data.cache:set:45 - Cached 4668 rows key=4fa02ad6c942


98 ETFs, 90 with Yahoo data


,asset_class,name,ticker,wrapper,fund_manager,has_data,n_days,ytd_return,sharpe_1Y,ann_return_1Y,...,sharpe_3Y,ann_return_3Y,max_dd_3Y,vol_3Y,sharpe_5Y,ann_return_5Y,max_dd_5Y,vol_5Y,corr_book,corr_n
0,All Country World,iShares MSCI ACWI UCITS ETF,ISAC.L,UCITS (LSE),iShares,True,1577,0.101938,2.089687,0.286922,...,1.441196,0.204306,-0.165600,0.135394,0.791497,0.117310,-0.252319,0.155445,0.399929,283
1,Artificial Intelligence,Xtrackers Artificial Intelligence & Big Data U...,XAID,?,Xtrackers,False,0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
2,Asia ex-Japan,iShares MSCI All Country Asia ex Japan ETF,AAXJ,US,iShares,True,1561,0.242788,2.312403,0.545215,...,1.350749,0.266164,-0.196257,0.187810,0.622989,0.108974,-0.387568,0.197122,0.666642,283
3,Asia High Yield USD Corporate Bonds *,iShares USD Asia High Yield Bond ETF,AHYG.SI,SG,iShares,False,0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
4,Australia,iShares MSCI Australia ETF,EWA,US,iShares,True,1561,0.104461,1.297645,0.226012,...,0.801609,0.141867,-0.222865,0.187422,0.628411,0.109493,-0.238233,0.195863,0.709183,283
5,Aerospace & Defense,Invesco Aerospace & Defense ETF,PPA,US,Invesco,True,1561,0.104134,1.524783,0.304916,...,1.546803,0.291656,-0.188185,0.175459,1.191409,0.220938,-0.188185,0.181397,0.667220,283
6,Battery Value-chain,L&G Battery Value-Chain UCITS ETF,BATT,?,LGIM,True,1561,0.221934,2.500766,1.040370,...,0.714414,0.175688,-0.432516,0.282350,0.465148,0.098169,-0.540348,0.294443,0.667746,283
7,Biotechnology,iShares Biotechnology ETF,IBB,US,iShares,True,1561,-0.012310,1.585889,0.350155,...,0.519671,0.086849,-0.258801,0.198001,0.324873,0.048200,-0.364711,0.218168,0.524787,283
8,Bitcoin (Accredited Investors only),Fidelity® Wise Origin® Bitcoin Fund,FBTC,?,Fidelity,True,575,-0.062571,-0.353863,-0.216277,...,0.795431,0.315487,-0.465991,0.504064,0.795431,0.315487,-0.465991,0.504064,0.669861,283
9,Blockchain,Invesco CoinShares Global Blockchain UCITS ETF,BCHN.L,UCITS (LSE),Invesco,True,1575,0.201512,1.144133,0.475089,...,0.995009,0.370155,-0.369585,0.394984,0.325883,0.052735,-0.572249,0.384653,0.376885,283


## Render HTML report

In [4]:
out = render_etf_explorer_report(
    explorer_df,
    REPORT_PATH,
    target_ann_return=TARGET_ANN_RETURN,
)
print(f'Wrote {out.resolve()}')

Wrote C:\Users\Dalva\src\project-hail-mary\reports\etf_explorer.html
